In [14]:
# ============================================================
# W0 EXACT POPULATION AUDIT — CORRECT GROUP MAPPING
# ============================================================

import os
import json
import numpy as np
import pandas as pd

FINAL_JSON = "/content/final_data_1.json"
FOLD_CSV = "/content/cmose_h1_h2_fold_protocol.csv"
RESULT_DIR = "/content/wu_global_i3d_results"

os.makedirs(RESULT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. Load CMOSE JSON
# ------------------------------------------------------------

with open(FINAL_JSON, "r") as f:
    raw = json.load(f)

print("Raw JSON records:", len(raw))

# ------------------------------------------------------------
# 2. Parse clips
# ------------------------------------------------------------

rows = []

for clip_id, record in raw.items():

    if not isinstance(record, dict):
        continue

    label = record.get("label")
    emb = record.get("embeds")

    if label is None or emb is None:
        continue

    try:
        emb = np.asarray(emb, dtype=np.float32)
    except Exception:
        continue

    if emb.ndim != 1 or len(emb) != 1024:
        continue

    parts = str(clip_id).split("_")

    # Expected:
    # video5_146_person5
    #  ^       ^      ^
    # session clip   person

    if len(parts) != 3:
        continue

    session = parts[0]
    clip_index = parts[1]
    person = parts[2]

    group = f"{session}_{person}"

    rows.append({
        "clip_id": str(clip_id),
        "session": session,
        "clip_index": int(clip_index),
        "person": person,
        "group": group,
        "label": str(label),
        "embedding": emb,
    })

w0_df = pd.DataFrame(rows)

print("\nW0 before protocol filtering")
print("-----------------------------")
print("Valid 1024-D labelled clips:", len(w0_df))
print("Unique groups:", w0_df["group"].nunique())

print("\nLabel counts:")
print(w0_df["label"].value_counts())

# ------------------------------------------------------------
# 3. Load protocol
# ------------------------------------------------------------

fold_df = pd.read_csv(FOLD_CSV)

fold_df["group"] = fold_df["group"].astype(str)

eligible_groups = set(
    fold_df["group"].unique()
)

print("\nProtocol")
print("--------")
print("Protocol groups:", len(eligible_groups))

# ------------------------------------------------------------
# 4. Check mapping directly
# ------------------------------------------------------------

mapped_groups = set(
    w0_df["group"].unique()
)

print("JSON-derived groups:", len(mapped_groups))

intersection_groups = (
    mapped_groups & eligible_groups
)

print("Group intersection:", len(intersection_groups))
print("Groups only in JSON:", len(mapped_groups - eligible_groups))
print("Groups only in protocol:", len(eligible_groups - mapped_groups))

# ------------------------------------------------------------
# 5. Filter to exact eligible groups
# ------------------------------------------------------------

w0_df = w0_df[
    w0_df["group"].isin(eligible_groups)
].copy()

print("\nAfter eligible-group filtering")
print("-------------------------------")
print("Eligible clips:", len(w0_df))
print("Eligible groups:", w0_df["group"].nunique())

# ------------------------------------------------------------
# 6. Attach fold
# ------------------------------------------------------------

group_fold = (
    fold_df[["group", "fold"]]
    .drop_duplicates()
)

# Verify one fold per group
fold_per_group = (
    group_fold
    .groupby("group")["fold"]
    .nunique()
)

bad_fold_groups = fold_per_group[
    fold_per_group != 1
]

assert len(bad_fold_groups) == 0, (
    "Some groups have multiple fold assignments."
)

w0_df = w0_df.merge(
    group_fold,
    on="group",
    how="left",
    validate="many_to_one",
)

missing_fold = w0_df["fold"].isna().sum()

print("Missing fold assignments:", missing_fold)

assert missing_fold == 0

# ------------------------------------------------------------
# 7. Exact W0 clip list
# ------------------------------------------------------------

w0_clip_ids = (
    w0_df[
        [
            "clip_id",
            "group",
            "session",
            "person",
            "clip_index",
            "fold",
            "label",
        ]
    ]
    .sort_values(
        ["fold", "group", "clip_index"]
    )
    .reset_index(drop=True)
)

clip_path = os.path.join(
    RESULT_DIR,
    "W0_exact_eligible_clip_ids.csv"
)

w0_clip_ids.to_csv(
    clip_path,
    index=False
)

print("\nSaved:")
print(clip_path)

# ------------------------------------------------------------
# 8. Fold summary
# ------------------------------------------------------------

fold_summary = (
    w0_df
    .groupby("fold")
    .agg(
        n_clips=("clip_id", "count"),
        n_groups=("group", "nunique"),
    )
    .reset_index()
    .sort_values("fold")
)

print("\nW0 FOLD SUMMARY")
print("================")
print(fold_summary.to_string(index=False))

# ------------------------------------------------------------
# 9. Group summary
# ------------------------------------------------------------

group_summary = (
    w0_df
    .groupby(["fold", "group"])
    .agg(
        n_clips=("clip_id", "count"),
        n_labels=("label", "nunique"),
    )
    .reset_index()
)

group_summary_path = os.path.join(
    RESULT_DIR,
    "W0_group_summary.csv"
)

group_summary.to_csv(
    group_summary_path,
    index=False
)

# ------------------------------------------------------------
# 10. Basic integrity checks
# ------------------------------------------------------------

print("\nINTEGRITY CHECKS")
print("================")

print(
    "All eligible groups mapped:",
    mapped_groups >= eligible_groups
)

print(
    "All W0 groups have folds:",
    missing_fold == 0
)

print(
    "82-group protocol matched:",
    w0_df["group"].nunique() == len(eligible_groups)
)

print(
    "No duplicate ClipIDs:",
    w0_df["clip_id"].is_unique
)

print(
    "Expected 5 folds:",
    sorted(w0_df["fold"].unique().tolist())
)

print("\nAudit complete.")

Raw JSON records: 12197

W0 before protocol filtering
-----------------------------
Valid 1024-D labelled clips: 11902
Unique groups: 103

Label counts:
label
Engage              8245
Disengage           2170
Highly Engage       1145
Highly Disengage     342
Name: count, dtype: int64

Protocol
--------
Protocol groups: 82
JSON-derived groups: 103
Group intersection: 82
Groups only in JSON: 21
Groups only in protocol: 0

After eligible-group filtering
-------------------------------
Eligible clips: 11426
Eligible groups: 82
Missing fold assignments: 0

Saved:
/content/wu_global_i3d_results/W0_exact_eligible_clip_ids.csv

W0 FOLD SUMMARY
 fold  n_clips  n_groups
    1     2367        17
    2     2254        17
    3     2216        16
    4     2640        16
    5     1949        16

INTEGRITY CHECKS
All eligible groups mapped: True
All W0 groups have folds: True
82-group protocol matched: True
No duplicate ClipIDs: True
Expected 5 folds: [1, 2, 3, 4, 5]

Audit complete.


In [17]:
# ============================================================
# W0 — GLOBAL / NO-PERSONALIZATION I3D BASELINE
# FULL 5-FOLD RUN
#
# Uses:
#   - CMOSE final_data_1.json
#   - exact 82-group evaluation protocol
#   - exact 11,426 eligible clips
#   - same early/random support selection as M1/H5/H6
#   - K = 5, 10, 20
#   - group-disjoint 5-fold evaluation
#   - training-only StandardScaler
#   - global classifier; support clips NEVER adapt the model
#
# IMPORTANT:
# This is a feature-level CMOSE I3D global baseline, NOT an exact
# reproduction of Wu et al.'s raw-video ResNet-TCN/MocoRank model.
# ============================================================

import os
import json
import random
import warnings
import numpy as np
import pandas as pd

from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    recall_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

# ============================================================
# 0. CONFIGURATION
# ============================================================

FINAL_JSON = "/content/final_data_1.json"
FOLD_CSV = "/content/cmose_h1_h2_fold_protocol.csv"

RESULT_DIR = "/content/wu_global_i3d_results"
os.makedirs(RESULT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Experiment settings
# ------------------------------------------------------------

BASE_SEED = 42

N_CLASSES = 4

# Label order is fixed throughout the experiment.
LABEL_NAMES = [
    "Highly Disengage",
    "Disengage",
    "Engage",
    "Highly Engage",
]

LABEL_TO_ID = {
    "Highly Disengage": 0,
    "Disengage": 1,
    "Engage": 2,
    "Highly Engage": 3,
}

K_SHOTS = [5, 10, 20]
PROTOCOLS = ["early", "random"]

# ------------------------------------------------------------
# Training settings
#
# This is intentionally a modest global MLP on the released
# 1024-D I3D representation.
# ------------------------------------------------------------

HIDDEN_DIM = 256
DROPOUT = 0.20

BATCH_SIZE = 128
EPOCHS = 80

LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-3

# Print progress every N epochs
PRINT_EVERY = 10

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 72)
print("W0 FULL RUN")
print("=" * 72)
print("Device:", DEVICE)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("")

# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

def seed_everything(seed):
    seed = int(seed)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(BASE_SEED)

# ============================================================
# 2. EXACT M1/H5/H6 SUPPORT SELECTION
# ============================================================

def get_support_query(
    student_df,
    k_shot,
    protocol,
    rng=None,
):
    """
    Exact support/query logic used by the personalization
    experiments.

    early:
        first K clips after clip_index sorting.

    random:
        K clips sampled without replacement using the supplied
        NumPy Generator.

    query:
        every remaining clip.
    """

    ordered = (
        student_df
        .sort_values("clip_index")
        .reset_index(drop=True)
    )

    if len(ordered) <= k_shot:
        return None, None

    if protocol == "early":

        support_idx = np.arange(k_shot)

    elif protocol == "random":

        if rng is None:
            raise ValueError(
                "Random support selection requires rng."
            )

        support_idx = rng.choice(
            ordered.index.to_numpy(),
            size=k_shot,
            replace=False,
        )

    else:
        raise ValueError(
            f"Unknown protocol: {protocol}"
        )

    support = ordered.iloc[
        support_idx
    ].copy()

    support_ids = set(
        support["clip_id"].astype(str)
    )

    query = ordered[
        ~ordered["clip_id"].astype(str).isin(support_ids)
    ].copy()

    if len(support) != k_shot:
        raise AssertionError(
            f"Support size {len(support)} != K={k_shot}"
        )

    if len(query) == 0:
        raise AssertionError(
            "Query set is empty."
        )

    if (
        set(support["clip_id"])
        &
        set(query["clip_id"])
    ):
        raise AssertionError(
            "Support/query overlap detected."
        )

    return support, query


def make_exact_eval_rng(
    seed,
    fold_id,
    k,
    group,
    protocol,
):
    """
    EXACT seed formula used by M1/H5/H6.

    local_seed =
        seed
        + fold_id * 10000
        + k * 100
        + sum(ord(ch) for ch in group)
        + (1 if protocol == "early" else 0)
    """

    local_seed = (
        seed
        + fold_id * 10000
        + k * 100
        + sum(ord(ch) for ch in group)
        + (1 if protocol == "early" else 0)
    )

    return np.random.default_rng(local_seed)


# ============================================================
# 3. MODEL
# ============================================================

class GlobalI3DClassifier(nn.Module):

    def __init__(
        self,
        input_dim=1024,
        hidden_dim=256,
        num_classes=4,
        dropout=0.20,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)


# ============================================================
# 4. TRAINING FUNCTION
# ============================================================

def train_global_model(
    X_train,
    y_train,
    seed,
    fold_id,
):
    """
    Train W0 global model using ONLY the training folds.

    Class-weighted CE is computed exclusively from the training
    fold. No test/student information enters training.
    """

    seed_everything(int(seed) + int(fold_id))

    X_train = np.asarray(
        X_train,
        dtype=np.float32,
    )

    y_train = np.asarray(
        y_train,
        dtype=np.int64,
    )

    # --------------------------------------------------------
    # Class weights from TRAINING DATA ONLY
    # --------------------------------------------------------

    counts = np.bincount(
        y_train,
        minlength=N_CLASSES,
    ).astype(np.float64)

    # Inverse-frequency weighting.
    # Zero-count classes receive zero weight.
    weights = np.zeros(
        N_CLASSES,
        dtype=np.float64,
    )

    nonzero = counts > 0

    if nonzero.any():
        weights[nonzero] = (
            len(y_train)
            /
            (
                N_CLASSES
                * counts[nonzero]
            )
        )

    # Normalize mean nonzero weight to 1.
    nz = weights > 0

    if nz.any():
        weights[nz] /= weights[nz].mean()

    class_weights = torch.tensor(
        weights,
        dtype=torch.float32,
        device=DEVICE,
    )

    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------

    X_tensor = torch.from_numpy(X_train)
    y_tensor = torch.from_numpy(y_train)

    dataset = TensorDataset(
        X_tensor,
        y_tensor,
    )

    generator = torch.Generator()
    generator.manual_seed(int(seed) + int(fold_id))

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = GlobalI3DClassifier(
        input_dim=X_train.shape[1],
        hidden_dim=HIDDEN_DIM,
        num_classes=N_CLASSES,
        dropout=DROPOUT,
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS,
    )

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    model.train()

    for epoch in range(1, EPOCHS + 1):

        running_loss = 0.0
        n_seen = 0

        for xb, yb in loader:

            xb = xb.to(
                DEVICE,
                non_blocking=True,
            )

            yb = yb.to(
                DEVICE,
                non_blocking=True,
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(xb)

            loss = criterion(
                logits,
                yb,
            )

            loss.backward()

            optimizer.step()

            batch_n = len(yb)

            running_loss += (
                loss.item() * batch_n
            )

            n_seen += batch_n

        scheduler.step()

        avg_loss = (
            running_loss / max(n_seen, 1)
        )

        if (
            epoch == 1
            or epoch % PRINT_EVERY == 0
            or epoch == EPOCHS
        ):
            print(
                f"      Epoch {epoch:03d}/{EPOCHS} "
                f"loss={avg_loss:.5f} "
                f"lr={scheduler.get_last_lr()[0]:.2e}"
            )

    return model


# ============================================================
# 5. PREDICTION FUNCTION
# ============================================================

@torch.no_grad()
def predict_model(
    model,
    X,
    batch_size=512,
):
    model.eval()

    X = np.asarray(
        X,
        dtype=np.float32,
    )

    tensor = torch.from_numpy(X)

    loader = DataLoader(
        tensor,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    predictions = []

    for xb in loader:

        xb = xb.to(
            DEVICE,
            non_blocking=True,
        )

        logits = model(xb)

        pred = torch.argmax(
            logits,
            dim=1,
        )

        predictions.append(
            pred.cpu().numpy()
        )

    if not predictions:
        return np.array([], dtype=np.int64)

    return np.concatenate(
        predictions
    )


# ============================================================
# 6. LOAD EXACT W0 POPULATION
# ============================================================

print("=" * 72)
print("LOADING DATA")
print("=" * 72)

with open(FINAL_JSON, "r") as f:
    raw = json.load(f)

print("Raw JSON records:", len(raw))

fold_df = pd.read_csv(
    FOLD_CSV
)

fold_df["group"] = (
    fold_df["group"]
    .astype(str)
)

eligible_groups = set(
    fold_df["group"].unique()
)

print(
    "Protocol groups:",
    len(eligible_groups)
)

rows = []

for clip_id, record in raw.items():

    if not isinstance(record, dict):
        continue

    label = record.get("label")
    emb = record.get("embeds")

    if label is None or emb is None:
        continue

    if label not in LABEL_TO_ID:
        continue

    try:
        emb = np.asarray(
            emb,
            dtype=np.float32,
        )
    except Exception:
        continue

    if emb.ndim != 1 or len(emb) != 1024:
        continue

    parts = str(clip_id).split("_")

    if len(parts) != 3:
        continue

    session = parts[0]
    clip_index = parts[1]
    person = parts[2]

    try:
        clip_index_int = int(
            clip_index
        )
    except Exception:
        continue

    group = (
        f"{session}_{person}"
    )

    # EXACT 82-GROUP POPULATION
    if group not in eligible_groups:
        continue

    rows.append({
        "clip_id": str(clip_id),
        "session": session,
        "clip_index": clip_index_int,
        "person": person,
        "group": group,
        "label": str(label),
        "label_id": LABEL_TO_ID[label],
        "embedding": emb,
    })

df = pd.DataFrame(rows)

if len(df) == 0:
    raise RuntimeError(
        "No W0 clips were loaded."
    )

print(
    "Eligible W0 clips:",
    len(df)
)

print(
    "Eligible W0 groups:",
    df["group"].nunique()
)

assert len(df) == 11426, (
    f"Expected 11,426 eligible clips, "
    f"got {len(df)}"
)

assert df["group"].nunique() == 82

assert df["clip_id"].is_unique

# ------------------------------------------------------------
# Attach fold
# ------------------------------------------------------------

group_fold = (
    fold_df[
        ["group", "fold"]
    ]
    .drop_duplicates()
)

df = df.merge(
    group_fold,
    on="group",
    how="left",
    validate="many_to_one",
)

assert df["fold"].notna().all()

df["fold"] = (
    df["fold"]
    .astype(int)
)

# ------------------------------------------------------------
# Verify fold structure
# ------------------------------------------------------------

fold_check = (
    df
    .groupby("fold")
    .agg(
        n_clips=("clip_id", "count"),
        n_groups=("group", "nunique"),
    )
    .reset_index()
    .sort_values("fold")
)

print("\nW0 population by fold:")
print(
    fold_check.to_string(
        index=False
    )
)

assert fold_check["fold"].tolist() == [
    1, 2, 3, 4, 5
]

# ============================================================
# 7. SAVE EXACT POPULATION
# ============================================================

population_path = os.path.join(
    RESULT_DIR,
    "W0_FULL_exact_population.csv"
)

df[
    [
        "clip_id",
        "group",
        "session",
        "person",
        "clip_index",
        "fold",
        "label",
        "label_id",
    ]
].sort_values(
    ["fold", "group", "clip_index"]
).to_csv(
    population_path,
    index=False,
)

print(
    "\nSaved population:",
    population_path
)

# ============================================================
# 8. PREPARE EMBEDDING MATRIX
# ============================================================

X_all = np.stack(
    df["embedding"].values
).astype(np.float32)

y_all = df["label_id"].to_numpy(
    dtype=np.int64
)

print("\nEmbedding matrix:")
print("X:", X_all.shape)
print("y:", y_all.shape)

assert X_all.shape == (
    len(df),
    1024,
)

# ============================================================
# 9. FULL 5-FOLD EXPERIMENT
# ============================================================

print("\n" + "=" * 72)
print("STARTING FULL W0 EXPERIMENT")
print("=" * 72)

all_student_results = []
all_class_results = []
all_confusion_results = []
all_skipped_results = []

# ------------------------------------------------------------
# Outer folds
# ------------------------------------------------------------

for fold_id in sorted(
    df["fold"].unique()
):

    print("\n")
    print("#" * 72)
    print(
        f"FOLD {fold_id}/5"
    )
    print("#" * 72)

    train_mask = (
        df["fold"] != fold_id
    )

    test_mask = (
        df["fold"] == fold_id
    )

    train_df = df.loc[
        train_mask
    ].copy()

    test_df = df.loc[
        test_mask
    ].copy()

    print(
        "Train groups:",
        train_df["group"].nunique()
    )

    print(
        "Test groups:",
        test_df["group"].nunique()
    )

    print(
        "Train clips:",
        len(train_df)
    )

    print(
        "Test clips:",
        len(test_df)
    )

    # --------------------------------------------------------
    # Explicit group-disjointness check
    # --------------------------------------------------------

    train_groups = set(
        train_df["group"]
    )

    test_groups = set(
        test_df["group"]
    )

    overlap = (
        train_groups &
        test_groups
    )

    print(
        "Train/test group overlap:",
        len(overlap)
    )

    assert len(overlap) == 0

    # --------------------------------------------------------
    # Extract training features
    # --------------------------------------------------------

    X_train = np.stack(
        train_df["embedding"].values
    ).astype(np.float32)

    y_train = train_df[
        "label_id"
    ].to_numpy(
        dtype=np.int64
    )

    # --------------------------------------------------------
    # TRAINING-ONLY SCALER
    # --------------------------------------------------------

    scaler = StandardScaler()

    X_train_scaled = (
        scaler
        .fit_transform(X_train)
        .astype(np.float32)
    )

    # We never fit the scaler on test students.
    # Transform only.
    #
    # This is an important leakage check.

    # --------------------------------------------------------
    # Train global model
    # --------------------------------------------------------

    print("\nTraining W0 global model...")

    model = train_global_model(
        X_train_scaled,
        y_train,
        seed=BASE_SEED,
        fold_id=fold_id,
    )

    # --------------------------------------------------------
    # Prepare ALL test features once
    # --------------------------------------------------------

    X_test_all = np.stack(
        test_df["embedding"].values
    ).astype(np.float32)

    X_test_all_scaled = (
        scaler
        .transform(X_test_all)
        .astype(np.float32)
    )

    # Map clip_id -> scaled feature
    test_feature_map = {
        clip_id: X_test_all_scaled[i]
        for i, clip_id
        in enumerate(
            test_df["clip_id"].astype(str)
        )
    }

    # --------------------------------------------------------
    # Evaluate every held-out student
    # --------------------------------------------------------

    fold_student_count = 0

    for group in sorted(
        test_df["group"].unique()
    ):

        student_df = (
            test_df[
                test_df["group"] == group
            ]
            .copy()
            .sort_values("clip_index")
        )

        # ----------------------------------------------------
        # Evaluate all K × protocol conditions
        # ----------------------------------------------------

        for protocol in PROTOCOLS:

            for k in K_SHOTS:

                # ------------------------------------------------
                # Exact M1/H5/H6 random seed
                # ------------------------------------------------

                if protocol == "random":

                    rng = make_exact_eval_rng(
                        seed=BASE_SEED,
                        fold_id=fold_id,
                        k=k,
                        group=group,
                        protocol=protocol,
                    )

                else:
                    rng = None

                # ------------------------------------------------
                # Exact support/query partition
                # ------------------------------------------------

                support, query = (
                    get_support_query(
                        student_df,
                        k_shot=k,
                        protocol=protocol,
                        rng=rng,
                    )
                )

                if support is None:
                    all_skipped_results.append({
                        "fold": fold_id,
                        "group": group,
                        "protocol": protocol,
                        "k": k,
                        "reason": "insufficient_clips",
                        "n_clips": len(student_df),
                    })
                    continue

                if query is None or len(query) == 0:
                    all_skipped_results.append({
                        "fold": fold_id,
                        "group": group,
                        "protocol": protocol,
                        "k": k,
                        "reason": "empty_query",
                        "n_clips": len(student_df),
                    })
                    continue

                # ------------------------------------------------
                # Explicit support/query integrity
                # ------------------------------------------------

                support_ids = set(
                    support["clip_id"].astype(str)
                )

                query_ids = set(
                    query["clip_id"].astype(str)
                )

                assert len(support_ids) == k

                assert len(
                    support_ids & query_ids
                ) == 0

                assert (
                    len(support_ids)
                    + len(query_ids)
                    == len(student_df)
                )

                # ------------------------------------------------
                # IMPORTANT:
                # W0 DOES NOT USE SUPPORT TO ADAPT THE MODEL.
                #
                # We construct support exactly like M1/H5/H6
                # solely so the W0 query set is IDENTICAL.
                # ------------------------------------------------

                query_ids_ordered = (
                    query["clip_id"]
                    .astype(str)
                    .tolist()
                )

                X_query = np.stack([
                    test_feature_map[cid]
                    for cid in query_ids_ordered
                ]).astype(np.float32)

                y_query = query[
                    "label_id"
                ].to_numpy(
                    dtype=np.int64
                )

                # ------------------------------------------------
                # Global model prediction
                # ------------------------------------------------

                y_pred = predict_model(
                    model,
                    X_query,
                )

                # ------------------------------------------------
                # Metrics
                # ------------------------------------------------

                acc = accuracy_score(
                    y_query,
                    y_pred,
                )

                bal_acc = balanced_accuracy_score(
                    y_query,
                    y_pred,
                )

                macro_f1 = f1_score(
                    y_query,
                    y_pred,
                    labels=np.arange(N_CLASSES),
                    average="macro",
                    zero_division=0,
                )

                recalls = recall_score(
                    y_query,
                    y_pred,
                    labels=np.arange(N_CLASSES),
                    average=None,
                    zero_division=0,
                )

                # Make sure recalls always has four entries.
                if len(recalls) != N_CLASSES:
                    full_recalls = np.zeros(
                        N_CLASSES,
                        dtype=np.float64,
                    )

                    for class_id in range(
                        N_CLASSES
                    ):
                        mask = (
                            y_query
                            == class_id
                        )

                        if mask.sum() > 0:
                            full_recalls[
                                class_id
                            ] = recall_score(
                                y_query,
                                y_pred,
                                labels=[class_id],
                                average=None,
                                zero_division=0,
                            )[0]

                    recalls = full_recalls

                # ------------------------------------------------
                # Student-level result
                # ------------------------------------------------

                all_student_results.append({
                    "fold": fold_id,
                    "group": group,
                    "protocol": protocol,
                    "k": k,

                    "n_total_clips": len(
                        student_df
                    ),
                    "n_support": len(
                        support
                    ),
                    "n_query": len(
                        query
                    ),

                    "accuracy": acc,
                    "balanced_accuracy": bal_acc,
                    "macro_f1": macro_f1,

                    "recall_highly_disengage": recalls[0],
                    "recall_disengage": recalls[1],
                    "recall_engage": recalls[2],
                    "recall_highly_engage": recalls[3],
                })

                # ------------------------------------------------
                # Per-class results
                # ------------------------------------------------

                for class_id, class_name in enumerate(
                    LABEL_NAMES
                ):

                    true_count = int(
                        np.sum(
                            y_query
                            == class_id
                        )
                    )

                    pred_count = int(
                        np.sum(
                            y_pred
                            == class_id
                        )
                    )

                    all_class_results.append({
                        "fold": fold_id,
                        "group": group,
                        "protocol": protocol,
                        "k": k,
                        "class_id": class_id,
                        "class_name": class_name,
                        "true_count": true_count,
                        "predicted_count": pred_count,
                        "recall": recalls[class_id],
                    })

                # ------------------------------------------------
                # Confusion matrix
                # ------------------------------------------------

                cm = confusion_matrix(
                    y_query,
                    y_pred,
                    labels=np.arange(
                        N_CLASSES
                    ),
                )

                for true_class in range(
                    N_CLASSES
                ):

                    for pred_class in range(
                        N_CLASSES
                    ):

                        all_confusion_results.append({
                            "fold": fold_id,
                            "group": group,
                            "protocol": protocol,
                            "k": k,
                            "true_class_id": true_class,
                            "true_class": LABEL_NAMES[
                                true_class
                            ],
                            "pred_class_id": pred_class,
                            "pred_class": LABEL_NAMES[
                                pred_class
                            ],
                            "count": int(
                                cm[
                                    true_class,
                                    pred_class
                                ]
                            ),
                        })

                fold_student_count += 1

    print(
        f"\nCompleted fold {fold_id}. "
        f"Student-condition evaluations: "
        f"{fold_student_count}"
    )

    # --------------------------------------------------------
    # Free GPU memory between folds
    # --------------------------------------------------------

    del model
    del scaler

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ============================================================
# 10. CONVERT RESULTS TO DATAFRAMES
# ============================================================

student_results = pd.DataFrame(
    all_student_results
)

class_results = pd.DataFrame(
    all_class_results
)

confusion_results = pd.DataFrame(
    all_confusion_results
)

skipped_results = pd.DataFrame(
    all_skipped_results
)

print("\n" + "=" * 72)
print("RAW RESULT COUNTS")
print("=" * 72)

print(
    "Student-condition rows:",
    len(student_results)
)

print(
    "Per-class rows:",
    len(class_results)
)

print(
    "Confusion rows:",
    len(confusion_results)
)

print(
    "Skipped rows:",
    len(skipped_results)
)

# ============================================================
# 11. SAVE STUDENT-LEVEL RESULTS
# ============================================================

student_path = os.path.join(
    RESULT_DIR,
    "W0_FULL_per_student.csv"
)

student_results.to_csv(
    student_path,
    index=False,
)

class_path = os.path.join(
    RESULT_DIR,
    "W0_FULL_per_class.csv"
)

class_results.to_csv(
    class_path,
    index=False,
)

confusion_path = os.path.join(
    RESULT_DIR,
    "W0_FULL_confusion.csv"
)

confusion_results.to_csv(
    confusion_path,
    index=False,
)

skipped_path = os.path.join(
    RESULT_DIR,
    "W0_FULL_skipped.csv"
)

skipped_results.to_csv(
    skipped_path,
    index=False,
)

print("\nSaved:")
print(student_path)
print(class_path)
print(confusion_path)
print(skipped_path)

# ============================================================
# 12. AGGREGATE BY CONDITION
# ============================================================

if len(student_results) == 0:
    raise RuntimeError(
        "No student-level results were produced."
    )

metric_cols = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "recall_highly_disengage",
    "recall_disengage",
    "recall_engage",
    "recall_highly_engage",
]

summary = (
    student_results
    .groupby(
        ["protocol", "k"]
    )[metric_cols]
    .agg(
        ["mean", "std"]
    )
    .reset_index()
)

# Flatten column names
summary.columns = [
    "_".join(
        [str(x) for x in col if str(x) != ""]
    ).rstrip("_")
    for col in summary.columns
]

# ============================================================
# 13. ALSO COMPUTE FOLD-LEVEL MEANS
# ============================================================

fold_summary = (
    student_results
    .groupby(
        ["fold", "protocol", "k"]
    )[metric_cols]
    .mean()
    .reset_index()
)

fold_summary_path = os.path.join(
    RESULT_DIR,
    "W0_FULL_fold_summary.csv"
)

fold_summary.to_csv(
    fold_summary_path,
    index=False,
)

# ============================================================
# 14. CLEAN HUMAN-READABLE SUMMARY
# ============================================================

clean_summary = (
    student_results
    .groupby(
        ["protocol", "k"]
    )
    .agg(
        n_students=(
            "group",
            "nunique",
        ),
        n_evaluations=(
            "group",
            "count",
        ),

        accuracy_mean=(
            "accuracy",
            "mean",
        ),
        accuracy_std=(
            "accuracy",
            "std",
        ),

        balanced_accuracy_mean=(
            "balanced_accuracy",
            "mean",
        ),
        balanced_accuracy_std=(
            "balanced_accuracy",
            "std",
        ),

        macro_f1_mean=(
            "macro_f1",
            "mean",
        ),
        macro_f1_std=(
            "macro_f1",
            "std",
        ),

        highly_disengage_recall_mean=(
            "recall_highly_disengage",
            "mean",
        ),
        disengage_recall_mean=(
            "recall_disengage",
            "mean",
        ),
        engage_recall_mean=(
            "recall_engage",
            "mean",
        ),
        highly_engage_recall_mean=(
            "recall_highly_engage",
            "mean",
        ),
    )
    .reset_index()
)

summary_path = os.path.join(
    RESULT_DIR,
    "W0_FULL_summary.csv"
)

clean_summary.to_csv(
    summary_path,
    index=False,
)

# ============================================================
# 15. VERIFY EXPECTED NUMBER OF EVALUATIONS
# ============================================================

expected_evaluations = (
    82 * 6
)

actual_evaluations = len(
    student_results
)

print("\n" + "=" * 72)
print("FINAL INTEGRITY CHECKS")
print("=" * 72)

print(
    "Expected student-condition evaluations:",
    expected_evaluations,
)

print(
    "Actual student-condition evaluations:",
    actual_evaluations,
)

if actual_evaluations == expected_evaluations:
    print(
        "PASS: all 82 students × 6 conditions evaluated."
    )
else:
    print(
        "WARNING: some student-condition evaluations are missing."
    )

print(
    "\nUnique students:",
    student_results["group"].nunique()
)

print(
    "Unique folds:",
    student_results["fold"].nunique()
)

print(
    "Protocols:",
    sorted(
        student_results["protocol"].unique()
    )
)

print(
    "K values:",
    sorted(
        student_results["k"].unique()
    )
)

# ------------------------------------------------------------
# Check each student has exactly 6 conditions
# ------------------------------------------------------------

condition_counts = (
    student_results
    .groupby("group")
    .size()
)

print(
    "\nStudents with exactly 6 conditions:",
    int(
        (condition_counts == 6).sum()
    ),
    "/",
    len(condition_counts),
)

assert (
    condition_counts == 6
).all(), (
    "At least one student does not have "
    "all six evaluation conditions."
)

# ------------------------------------------------------------
# Check each fold has expected groups
# ------------------------------------------------------------

student_fold_counts = (
    student_results
    .groupby("fold")["group"]
    .nunique()
    .sort_index()
)

print("\nStudents per fold:")
print(
    student_fold_counts.to_string()
)

# ------------------------------------------------------------
# Check support sizes
# ------------------------------------------------------------

support_sizes = (
    student_results["n_support"]
    .value_counts()
    .sort_index()
)

print("\nSupport-size counts:")
print(
    support_sizes.to_string()
)

assert set(
    student_results["n_support"].unique()
) == set(K_SHOTS)

# ------------------------------------------------------------
# Check query is always non-empty
# ------------------------------------------------------------

assert (
    student_results["n_query"] > 0
).all()

print(
    "\nAll query sets non-empty: PASS"
)

# ============================================================
# 16. PRINT FINAL RESULTS
# ============================================================

print("\n" + "=" * 72)
print("W0 FINAL SUMMARY")
print("=" * 72)

display(
    clean_summary.round(4)
)

print("\nFold-level summary:")
display(
    fold_summary.round(4)
)

# ============================================================
# 17. SAVE A TEXT REPORT
# ============================================================

report_path = os.path.join(
    RESULT_DIR,
    "W0_FULL_run_report.txt"
)

with open(
    report_path,
    "w"
) as f:

    f.write(
        "W0 GLOBAL / NO-PERSONALIZATION I3D BASELINE\n"
    )
    f.write("=" * 72 + "\n\n")

    f.write(
        f"Raw JSON records: {len(raw)}\n"
    )

    f.write(
        f"Eligible clips: {len(df)}\n"
    )

    f.write(
        f"Eligible groups: {df['group'].nunique()}\n"
    )

    f.write(
        f"Device: {DEVICE}\n"
    )

    f.write(
        f"Epochs: {EPOCHS}\n"
    )

    f.write(
        f"Learning rate: {LEARNING_RATE}\n"
    )

    f.write(
        f"Weight decay: {WEIGHT_DECAY}\n"
    )

    f.write(
        f"Base seed: {BASE_SEED}\n\n"
    )

    f.write(
        "FINAL SUMMARY\n"
    )
    f.write(
        clean_summary.to_string(
            index=False
        )
    )

    f.write("\n\nFOLD SUMMARY\n")
    f.write(
        fold_summary.to_string(
            index=False
        )
    )

    f.write(
        "\n\nSkipped evaluations:\n"
    )
    f.write(
        str(len(skipped_results))
    )

print(
    "\nSaved report:",
    report_path
)

# ============================================================
# 18. DONE
# ============================================================

print("\n" + "=" * 72)
print("W0 FULL RUN COMPLETE")
print("=" * 72)

print("\nMain result file:")
print(summary_path)

print("\nStudent-level result file:")
print(student_path)

print("\nPer-class result file:")
print(class_path)

print("\nConfusion matrix file:")
print(confusion_path)

print("\nFold summary:")
print(fold_summary_path)

print("\nRun report:")
print(report_path)

print("\nYou can now compare W0 directly against M1/H5/H6.")

W0 FULL RUN
Device: cpu
CUDA available: False

LOADING DATA
Raw JSON records: 12197
Protocol groups: 82
Eligible W0 clips: 11426
Eligible W0 groups: 82

W0 population by fold:
 fold  n_clips  n_groups
    1     2367        17
    2     2254        17
    3     2216        16
    4     2640        16
    5     1949        16

Saved population: /content/wu_global_i3d_results/W0_FULL_exact_population.csv

Embedding matrix:
X: (11426, 1024)
y: (11426,)

STARTING FULL W0 EXPERIMENT


########################################################################
FOLD 1/5
########################################################################
Train groups: 65
Test groups: 17
Train clips: 9059
Test clips: 2367
Train/test group overlap: 0

Training W0 global model...
      Epoch 001/80 loss=1.03315 lr=5.00e-04
      Epoch 010/80 loss=0.31445 lr=4.81e-04
      Epoch 020/80 loss=0.13938 lr=4.27e-04
      Epoch 030/80 loss=0.07685 lr=3.46e-04
      Epoch 040/80 loss=0.04526 lr=2.50e-04
      Epoch 050/

,protocol,k,n_students,n_evaluations,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std,highly_disengage_recall_mean,disengage_recall_mean,engage_recall_mean,highly_engage_recall_mean
0,early,5,82,82,0.6230,0.1646,0.4525,0.1479,0.3308,0.0961,0.0526,0.3020,0.7509,0.3695
1,early,10,82,82,0.6231,0.1655,0.4567,0.1521,0.3287,0.0987,0.0513,0.3004,0.7515,0.3641
2,early,20,82,82,0.6251,0.1711,0.4650,0.1605,0.3280,0.1029,0.0557,0.2841,0.7534,0.3700
3,random,5,82,82,0.6230,0.1625,0.4466,0.1434,0.3319,0.0962,0.0504,0.3030,0.7513,0.3760
4,random,10,82,82,0.6260,0.1646,0.4470,0.1551,0.3342,0.0974,0.0519,0.3084,0.7512,0.3716
5,random,20,82,82,0.6254,0.1693,0.4516,0.1494,0.3331,0.0983,0.0540,0.3004,0.7520,0.3799



Fold-level summary:


,fold,protocol,k,accuracy,balanced_accuracy,macro_f1,recall_highly_disengage,recall_disengage,recall_engage,recall_highly_engage
0,1,early,5,0.6374,0.4574,0.3209,0.0749,0.1606,0.8268,0.3929
1,1,early,10,0.6374,0.4798,0.3169,0.0681,0.1567,0.8327,0.3897
2,1,early,20,0.6455,0.4768,0.3119,0.0657,0.1481,0.8400,0.3825
3,1,random,5,0.6361,0.4379,0.3187,0.0658,0.1560,0.8229,0.3807
4,1,random,10,0.6398,0.4399,0.3215,0.0651,0.1623,0.8250,0.3808
5,1,random,20,0.6300,0.4433,0.3149,0.0735,0.1361,0.8209,0.3964
6,2,early,5,0.6366,0.4635,0.3277,0.0997,0.3096,0.7656,0.2795
7,2,early,10,0.6381,0.4685,0.3302,0.1000,0.3157,0.7670,0.2899
8,2,early,20,0.6498,0.5142,0.3369,0.1235,0.3336,0.7739,0.2888
9,2,random,5,0.6349,0.4505,0.3260,0.0983,0.3032,0.7663,0.2837



Saved report: /content/wu_global_i3d_results/W0_FULL_run_report.txt

W0 FULL RUN COMPLETE

Main result file:
/content/wu_global_i3d_results/W0_FULL_summary.csv

Student-level result file:
/content/wu_global_i3d_results/W0_FULL_per_student.csv

Per-class result file:
/content/wu_global_i3d_results/W0_FULL_per_class.csv

Confusion matrix file:
/content/wu_global_i3d_results/W0_FULL_confusion.csv

Fold summary:
/content/wu_global_i3d_results/W0_FULL_fold_summary.csv

Run report:
/content/wu_global_i3d_results/W0_FULL_run_report.txt

You can now compare W0 directly against M1/H5/H6.
